# Middlewares 
* Middlewares gives developers tighter control over what happens inside an agent


**Middleware hooks into six points deciding what happens in middle of agentic loop:** 
1. Before agent run
2. After agent runs
3. Before model is called
4. After model is called
5. Before tool is called
6. After tool is called

<img src="../../assets/middleware_flow.png" width="1200" height="300">

This Nootebook runs through nine built-in middlewares end to end 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [28]:
# --- Core LangChain ---
from langchain.agents import create_agent
from langchain.tools import tool

# --- LangGraph (checkpointing, resuming) ---
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.agents.middleware import (
    SummarizationMiddleware,
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    ModelFallbackMiddleware,
    PIIMiddleware,
    TodoListMiddleware,
    LLMToolSelectorMiddleware,
    ToolRetryMiddleware,
    ToolErrorMiddleware,
    ModelRetryMiddleware,
    LLMToolEmulator,
    ContextEditingMiddleware,
    ClearToolUsesEdit,
)
from rich import print
import sys
sys.path.append('..')
from utils.helper import pretty_print_agent_output, print_messages

In [3]:
@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

In [4]:

@tool
def book_seats(movie_title: str, seat_count: int) -> str:
    """Book seats for a movie. Irreversible once confirmed."""
    return f"Booked {seat_count} seat(s) for {movie_title}."


In [5]:
@tool
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."


In [6]:
@tool
def check_order_status(booking_id: str) -> str:
    """Check the status of an existing booking."""
    return f"Booking {booking_id}: confirmed, 2 seats, Interstellar, 7:00 PM."


In [7]:
@tool
def get_refund_policy() -> str:
    """Get the cinema's refund policy -- exact wording, not to be paraphrased."""
    return "Refunds available up to 2 hours before showtime. No refunds after that."

In [8]:

@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."


In [9]:
tools = [check_showtimes, book_seats, cancel_booking, check_order_status, get_refund_policy, lookup_seat_map]

### 1 SummarizationMiddleware  
* Automatically summarize conversation history when approaching token limits, preserving recent messages.
* Usecase:
    * Long-running conversations that exceed context windows.
    * Multi-turn dialogues with extensive history.
    * Applications where preserving full conversation context matters.

##### Parameters
- **`model`** — summarizing text is itself a task that needs a LLM.A cheaper model works here.
- **`trigger`** — when summarization kicks in: 
    * token count
    * message count
    * fraction of the model's total context length (e.g. once the context is 80% full).
- **`keep`** — how much of the recent conversation to leave untouched after summarizing
    * fraction
    * token count
    * message count

In [17]:
summarizing_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=tools,
    middleware=[
        SummarizationMiddleware(
            model="openai:gpt-5-nano",
            trigger=("tokens", 300),
            keep=("messages", 1),
        )
    ]
)

In [ ]:
summarizing_agent.invoke({"messages": [("user", "Hi I am Shivam ")]})
summarizing_agent.invoke({"messages": [("user", "Who am I ? ")]})
result = summarizing_agent.invoke({"messages": [("user", "Is Interstellar showing tonight? also please make sure that you book me a ticket, refund me if it is not available,also share the refund policy for me to go through, also check my order status for book_1234")]})
pretty_print_agent_output(result)

🤖 AGENT OUTPUT

======================================================================

👤 User:

Here is a summary of the conversation to date:

## SESSION INTENT

Determine tonight's Interstellar showtime, book a ticket, refund if unavailable, share refund policy, and check 
order status for booking ID book_1234.

## SUMMARY

User requests:
- Check if Interstellar is showing tonight.
- Book a ticket.
- If tickets are not available, process a refund.
- Provide the refund policy for review.
- Check the order status for booking ID book_1234.

No actions executed yet. No location or theater specifics provided beyond movie title and booking ID.

## ARTIFACTS

None

## NEXT STEPS

- Check tonight's Interstellar showtimes at the user’s location/theater(s) on file.
- If seats are available, complete the ticket booking.
- If no seats are available, initiate refund process and provide refund policy details.
- Retrieve and report the status of order ID book_1234.

🤖 Assistant:

...

Args : {}

ID   : call_4QdlhwAIvQhDf9ZBJQKohN0F

🤖 Assistant:

Here are the results from our checks:

- Tonight's Interstellar showtimes: 7:00 PM and 10:15 PM.
- Your existing booking: ID book_1234 is confirmed for 2 seats at 7:00 PM.
- Refund policy: Refunds are available up to 2 hours before the showtime. No refunds after that.

What would you like to do next?
- Book additional seats for 7:00 PM or 10:15 PM (tell me how many seats and which showtime).
- Add seats to your current booking (this would be a new booking for the extra seats; I can proceed if you want).
- Cancel or modify a booking (provide the booking ID).
- Check status for another booking.

======================================================================

### 2 HumanInTheLoopMiddleware (HITL)
* Pauses agent execution to request human approval or input for critical decisions.  
* Example uses: 
    * Require user confirmation before executing destructive actions (deleting records, sending emails).
    * High-stakes operations requiring human approval (e.g. database writes, financial transactions).
    * Compliance workflows where human oversight is mandatory.
    * Long-running conversations where human feedback guides the agent.


In [ ]:
guarded_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}},
            description_prefix="The agent is about to call the `cancel_booking` tool, which is irreversible. Please review the request and choose an action: approve, edit, reject, or respond.",
            
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)

config = {'configurable':{'thread_id':'hitl-demo-live'}}

* Call the guarded agent that make a `cancel_booking` tool call.
* Because we added `HumanInTheLoopMiddleware` that interrupts on `cancel_booking`, the agent will pause and checkpoint the run so a human can approve, edit, reject, or respond before the tool executes.  

In [37]:
result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)
pretty_print_agent_output(result)

🤖 AGENT OUTPUT

======================================================================

👤 User:

Please cancel booking BK1042

🤖 Assistant:

...

Description: Tool execution requires approval

Tool: cancel_booking
Args: {'booking_id': 'BK1042'}

📋 Allowed decisions:

approve, edit, reject, respond

Interrupt ID: 57e285fa7402375183a05e3ed32846b1

======================================================================

* If intervention is needed, the middleware issues an interrupt that halts execution. 
* The **`graph state`** is saved using LangGraph’s persistence layer, so execution can pause safely and resume later.

In [43]:
state = guarded_agent.get_state(config)
print(state.tasks[0])

PregelTask(
    id='b8130241-8c80-c9c7-7ed2-8a4133d1d04b',
    name='HumanInTheLoopMiddleware.after_model',
    path=('__pregel_pull', 'HumanInTheLoopMiddleware.after_model'),
    error=None,
    interrupts=(
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'cancel_booking',
                        'args': {'booking_id': 'BK1042'},
                        'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: 
{'booking_id': 'BK1042'}"
                    }
                ],
                'review_configs': [
                    {
                        'action_name': 'cancel_booking',
                        'allowed_decisions': ['approve', 'edit', 'reject', 'respond']
                    }
                ]
            },
            id='57e285fa7402375183a05e3ed32846b1'
        ),
    ),
    state=None,
    result=None
)

**Resuming a paused HITL run**

* We resume the paused Human‑In‑The‑Loop run by invoking:
* This sends the human decision (here: "approve") back to the agent so it can continue and execute the guarded `cancel_booking` tool call.
>Command is a LangGraph control object (imported from langgraph.types) used to send control instructions into a paused agent run.<br>
>Passing Command(...) to agent.invoke (instead of a normal message) tells the agent to apply those decisions to the saved graph state and continue execution.

In [45]:
resumed_result = guarded_agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config=config)
pretty_print_agent_output(resumed_result)

🤖 AGENT OUTPUT

======================================================================

👤 User:

Please cancel booking BK1042

🤖 Assistant:

...

Args : {'booking_id': 'BK1042'}

ID   : call_AKEyxoTJ6vZR87gdeksGwWdW

🤖 Assistant:

Cancellation confirmed. Booking BK1042 has been cancelled.

Would you like me to:
- fetch the exact refund policy wording, or
- check the refund status for this booking, or
- help you rebook another movie?

======================================================================

**run_interactive_hitl_demo**
- Lightweight CLI helper for Human‑In‑The‑Loop (HITL).
- Checks whether the agent is currently paused awaiting human approval.
- Prompts a human to choose one of four decisions: approve / edit / reject / respond.
- Sends the chosen decision back to the agent via a resume Command, resuming execution.
- Prints the agent’s final response after the run resumes.
- Intended as a simple demo helper, not a full walkthrough of each decision type.

In [46]:
def run_interactive_hitl_demo(agent, config):
    """A genuinely interactive HITL loop -- ask out loud, type the answer, watch it apply live."""
    state = agent.get_state(config)
    if not state.next:
        print("Nothing is currently paused for approval.")
        return

    print("The agent wants to call a guarded tool. Choose a decision:")
    print("  1) approve  -- run it exactly as proposed")
    print("  2) edit     -- run it, but change the booking_id first")
    print("  3) reject   -- block it, with a reason sent back to the agent")
    print("  4) respond  -- answer a question instead of deciding on the action")

    choice = input("Type 1, 2, 3, or 4: ").strip()

    if choice == "1":
        decision = {"type": "approve"}
    elif choice == "2":
        new_id = input("New booking_id to use instead: ").strip()
        decision = {"type": "edit", "args": {"booking_id": new_id}}
    elif choice == "3":
        reason = input("Reason for rejecting: ").strip()
        decision = {"type": "reject", "message": reason}
    elif choice == "4":
        answer = input("Your response to the agent: ").strip()
        decision = {"type": "respond", "message": answer}
    else:
        print("Not a valid choice -- try again.")
        return

    resumed = agent.invoke(Command(resume={"decisions": [decision]}), config=config)
    print()
    print("Agent's final response:", resumed["messages"][-1].content)
    
config = {'configurable':{'thread_id':'hitl-demo'}}

In [ ]:
result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)
result['__interrupt__']

[Interrupt(value={'action_requests': [{'name': 'cancel_booking', 'args': {'booking_id': 'BK1042'}, 'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: {'booking_id': 'BK1042'}"}], 'review_configs': [{'action_name': 'cancel_booking', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='b1beca44f12758dc360483db4df7bd7a')]

In [50]:
run_interactive_hitl_demo(guarded_agent, config)

The agent wants to call a guarded tool. Choose a decision:

1) approve  -- run it exactly as proposed

2) edit     -- run it, but change the booking_id first

3) reject   -- block it, with a reason sent back to the agent

4) respond  -- answer a question instead of deciding on the action

Agent's final response: It looks like the cancellation didn’t complete. Current booking BK1042 is still confirmed.

Booking details:
- BK1042: 2 seats, Interstellar, 7:00 PM

Would you like me to try cancelling BK1042 again now? If you’d like, I can also pull the refund policy to confirm 
any eligible refund before I proceed.

### 3. ModelCallLimitMiddleware
* Enforces limits on how many LLM calls are allowed (per session / time window).  
* Example use: Cap LLM usage to control API costs (e.g., max 100 calls/day).

    * ModelCallLimitMiddleware(thread_limit=5, run_limit=2, exit_behavior="end").
    * low‑cost demo configuration that allows up to 5 calls per thread and up to 2 calls per run, stopping gracefully when exceeded.
thread_limit

* Limits that span a thread require a checkpointer (e.g., InMemorySaver) so counts persist between invokes.
* run_limit is ephemeral to the current invoke and does not require persistence.


In [52]:
call_limited_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(), 
    middleware=[
        ModelCallLimitMiddleware( 
            thread_limit=5,   # across the WHOLE conversation
            run_limit=2,       # per single .invoke() call
            exit_behavior="end",  # graceful stop, not an exception
        ),
    ],
)

In [ ]:
result = call_limited_agent.invoke(
    {"messages": [("user", "Can you tell me cinema's refund policy? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

result_invoke_2= call_limited_agent.invoke(
    {"messages": [("user", "cancel my booking B123? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

result_invoke3 = call_limited_agent.invoke(
    {"messages": [("user", "What all movies are being shown? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

result_invoke4 = call_limited_agent.invoke(
    {"messages": [("user", "Summarize my chat? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [59]:
pretty_print_agent_output(result_invoke4)

🤖 AGENT OUTPUT

======================================================================

👤 User:

Can you tell me cinema's refund policy?

🤖 Assistant:

...

👤 User:

Summarize my chat?

🤖 Assistant:

Model call limits exceeded: thread limit (5/5)

======================================================================

### 4. ModelFallbackMiddleware
* Switches to an alternate model automatically when the primary model fails or is unavailable.  
* Example use: Fallback from an expensive high-capacity model to a cheaper one on errors or budget constraints.

In [62]:
resilient_agent = create_agent(
    model="openai:gpt-5.5-haiku",
    tools=tools,
    middleware=[
        ModelFallbackMiddleware(
            "openai:gpt-5-nano",  
        ),
    ],
)
print("Fallback chain: openai:gpt-5.5-haiku -> openai:gpt-5-nano.")


Fallback chain: openai:gpt-5.5-haiku -> openai:gpt-5-nano.

In [63]:
result = resilient_agent.invoke( {"messages": [("user", "Summarize my chat? ")]},)

In [70]:
print(result['messages'][-1].response_metadata)

{
    'token_usage': {
        'completion_tokens': 622,
        'prompt_tokens': 283,
        'total_tokens': 905,
        'completion_tokens_details': {
            'accepted_prediction_tokens': 0,
            'audio_tokens': 0,
            'reasoning_tokens': 512,
            'rejected_prediction_tokens': 0
        },
        'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
    },
    'model_provider': 'openai',
    'model_name': 'gpt-5-nano-2025-08-07',
    'system_fingerprint': None,
    'id': 'chatcmpl-EDayGYPzz9fe0AvIF7QAAhPfm7FGv',
    'service_tier': 'default',
    'finish_reason': 'stop',
    'logprobs': None
}

### 5. ToolCallLimitMiddleware
* Control agent execution by limiting the number of tool calls, either globally across all tools or for specific tools.
* Example use: 
    * Limit web-scraper calls to avoid rate-limits or excessive costs (e.g., 5 scrapes per query).
    * Preventing excessive calls to expensive external APIs.
    * Limiting web searches or database queries.
    * Enforcing rate limits on specific tool usage.
    * Protecting against runaway agent loops

In [18]:
tool_limited_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    middleware=[
        ToolCallLimitMiddleware(run_limit=8),
        ToolCallLimitMiddleware(tool_name="cancel_booking", thread_limit=2, run_limit=1),  # tighter, one tool, whole conversation
    ],
)

config = {"configurable": {"thread_id": "tool-limits"}}

**Exceeding `thread_limit`**
* In above call we have `run_limit=1` i.e. in each invoke we can call `cancel_booking` tool just once.
* Also we have `thread_limit=2` so we can call `cancel_booking` tool 2 time with same `thread_id=tool-limits`
* In below example we invoke agent 3 times, in each invoke single call to `cancel_booking` tool will be made, and that will work fine as we have `run_limit=1`
* But once we invoke agent the 3rd time with same thread, it will exceed the `thread_limit=2` limit, get teh proper message like 'hit a system limit'

In [19]:
for i in range(3):
  result = tool_limited_agent.invoke({"messages": [("user", f"Please cancel my Booking with ID B{100+i} ? ")]}, config=config)
  print(result['messages'][-1].content)

The cancellation is complete. Booking ID B100 has been cancelled. If you need anything else, I can help.

Cancellation complete. Booking ID B101 has been cancelled.

If you’d like, I can check the refund policy or help with rebooking another movie. Would you like me to look up the
refund terms or assist with a new booking?

I tried to cancel Booking B102, but I hit a system limit on cancel operations. The current status is still:

- Booking B102: confirmed, 2 seats, Interstellar, 7:00 PM.

What would you like to do next?
- Retry the cancellation now (I can attempt again immediately).
- Retrieve the full refund policy to check eligibility before retrying.
- Help with rebooking a different movie or seats.

### 6. PII Detection
* Detect and handle Personally Identifiable Information (PII) in conversations using configurable strategies. 
* PII detection use-case:
    1. Healthcare and financial applications with compliance requirements.
    2. Customer service agents that need to sanitize logs.
    3. Any application handling sensitive user data.

**Built-in PII types:**
* **email:** Email addresses
* **credit_card:** Credit card numbers (validated with Luhn algorithm)
* **ip:** IP addresses (validated with stdlib)
* **mac_address:** MAC addresses
* **url:** URLs (both http/https and bare URLs)

**Strategies:**
* **block:** Raise an exception when PII is detected
* **redact:** Replace PII with [REDACTED_TYPE] placeholders
* **mask:** Partially mask PII (e.g., ****-****-****-1234 for credit card)
* **hash:** Replace PII with deterministic hash (e.g., <email_hash:a1b2c3d4>)

| **Strategy** | **Preserves Identity?** | **Best For**                            |
| ------------ | ----------------------- | --------------------------------------- |
| `block`      | N/A                     | Avoid PII completely                    |
| `redact`     | No                      | General compliance, log sanitization    |
| `mask`       | No                      | Human readability, customer service UIs |
| `hash`       | Yes (pseudonymous)      | Analytics, debugging                    |

**Configuration Options**

* `pii_type`: string --> required ['credit_card', 'email' ...]
* `detector`: Custom detector function or regex pattern.
* `apply_to_input`: boolean -> default:"True"
* `apply_to_output`: boolean -> default:"False"
* `apply_to_tool_results`: boolean -> default:"False"

In [20]:
pii_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=tools,
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)

In [23]:
result = pii_agent.invoke({
    "messages": [("user", "My email is shivam@example.com and my credit card is 4007000000027, can you check showtimes for Dune?")]
})

**This shoud REDACT my email ID in very first message (HumanMessage) itself before calling the LLM with actual value.**

In [26]:
print(result['messages'][0].content)

My email is [REDACTED_EMAIL] and my credit card is 4007000000027, can you check showtimes for Dune?

#### Custom PII Detection uing Regex
* Function that takes string content and detects custom pattern in the string

In [27]:
import re
def detect_booking_code(content: str) -> list[dict]:
    """Detect CineBot's own booking code format: BK followed by 4 digits."""
    matches = []
    for match in re.finditer(r"BK\d{4}", content):
        matches.append({"text": match.group(0), "start": match.start(), "end": match.end()})
    return matches

In [28]:
custom_pii_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=tools,
    middleware=[PIIMiddleware("booking_code", detector=detect_booking_code, strategy="mask")],
)

In [29]:
result = custom_pii_agent.invoke({
    "messages": [("user", "Can you check the status of my booking BK1044 for me?")]
})

In [30]:
pretty_print_agent_output(result)

🤖 AGENT OUTPUT

======================================================================

👤 User:

Can you check the status of my booking ****1044 for me?

🤖 Assistant:

...

Args : {'booking_id': '****1044'}

ID   : call_JnmRxfsneGywikDl61xJ7XBd

🤖 Assistant:

Here are the details for your booking ****1044:

- Status: Confirmed
- Movie: Interstellar
- Time: 7:00 PM
- Seats: 2

Would you like to cancel, modify the booking, or view the refund policy?

======================================================================

### 7. TodoListMiddleware
* Extracts or manages actionable tasks (to-dos) from agent interactions and keeps a task list.  
* Example use: 
    * Convert user requests into a prioritized checklist for multi-step workflows.
    * Complex multi-step tasks requiring coordination across multiple tools.
    * Long-running operations where progress visibility is important.

**This middleware automatically provides agents with a `write_todos` tool and system prompts to guide effective task planning.**

In [21]:
todo_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=tools,
    middleware=[TodoListMiddleware()],
    system_prompt="You are an assistant that can manage complex tasks. Always keep the user informed about their current to-do list."
)

**Invoke agent with multiple tasks**
* Below we asks Agent a complex query that requires it to maintain To-Do list and call `write_todos` tool provided from `TodoListMiddleware`

In [ ]:
result = todo_agent.invoke({
    "messages": [("user", "I want to plan a movie night: check what's showing, pick something good, and book 2 seats.")]
})
print_messages(result)

HumanMessage: I want to plan a movie night: check what's showing, pick something good, and book 2 seats.

================================================================================

Agent: AIMessage with tool calls:

Tool Call 1 Name: write_todos
Tool Call 1 Args: {'todos': [{'content': 'Ask user for cinema/location, date, and preferred time window for movie 
night.', 'status': 'in_progress'}, {'content': 'Fetch current showings at the chosen cinema for the selected 
date.', 'status': 'pending'}, {'content': 'Evaluate showings and suggest top candidate(s) based on ratings and 
preferences (genre, mood).', 'status': 'pending'}, {'content': 'Check availability and book 2 seats for the chosen 
showtime.', 'status': 'pending'}, {'content': 'Confirm booking details and share the reservation (booking ID, seat 
numbers, time).', 'status': 'pending'}]}

================================================================================

ToolMessage: Updated todo list to [{'content': 'Ask user for cinema/location, date, and preferred time window for 
movie night.', 'status': 'in_progress'}, {'content': 'Fetch current showings at the chosen cinema for the selected 
date.', 'status': 'pending'}, {'content': 'Evaluate showings and suggest top candidate(s) based on ratings and 
preferences (genre, mood).', 'status': 'pending'}, {'content': 'Check availability and book 2 seats for the chosen 
showtime.', 'status': 'pending'}, {'content': 'Confirm booking details and share the reservation (booking ID, seat 
numbers, time).', 'status': 'pending'}]

================================================================================

Agent: Great plan. I’m ready to handle the flow, and here’s where we are in your plan.

Current to-do list
- In progress: Ask you for cinema/location, date, and preferred time window for movie night.
- Pending: Fetch current showings at the chosen cinema for the selected date.
- Pending: Evaluate showings and suggest top candidate(s) based on ratings and preferences (genre, mood).
- Pending: Check availability and book 2 seats for the chosen showtime.
- Pending: Confirm booking details and share the reservation (booking ID, seat numbers, time).

To move forward, please provide:
- Which cinema or area would you like to watch at? (or I can pick a nearby option)
- Date for movie night and a preferred time window (e.g., anytime this Friday after 6pm)
- Any preferences: genre, mood, language, age rating, or accessibility needs
- Confirm: 2 seats is correct

If you’d like, I can also suggest top options once you share the cinema and date.

================================================================================

### 8. LLMToolSelectorMiddleware
* Use an LLM to intelligently select relevant tools before calling the main model.
* Example Use:
    * Agents with many tools (10+) where most aren’t relevant per query.
    * Reducing token usage by filtering irrelevant tools.
    * Improving model focus and accuracy.
> This middleware uses structured output to ask an LLM which tools are most relevant for the current query.<br> The structured output schema defines the available tool names and descriptions. <br>Model providers often add this structured output information to the system prompt behind the scenes

In [16]:
print('Tool List: ', [t.name for t in tools])

Tool List: 
['check_showtimes', 'book_seats', 'cancel_booking', 'check_order_status', 'get_refund_policy', 'lookup_seat_map']

**Funtion to see at Runtime, which tools are selected for a user query.**

In [17]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def show_tools(request, handler):
    print("\nTOOLS SENT TO MODEL:")
    print([tool.name for tool in request.tools])
    return handler(request)

In [25]:
selector_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=tools,
    middleware=[
        LLMToolSelectorMiddleware(
            model="openai:gpt-5-nano", 
            max_tools=2,
            always_include=["check_showtimes"],  # always kept, doesn't count against max_tools
        ),
        show_tools  # Too see which tools are loaded at runtime, after the selector has filtered them down to the most relevant ones.
    ],
)

**Invoke Agent to call Cancellation related Tools**
* Below middleware helps to reduce # Tools sent to agent, it sent only 3 Tools out of 6 based on User query.
* `check_showtimes` was always included.
* `cancel_booking` & `check_order_status` was loaded as `LLMToolSelectorMiddleware`  suggested these 2 tools will be required to fulfill this user query at runtime.

In [26]:
result = selector_agent.invoke({"messages": [("user", "Can you cancel my booking with ID B1234?")]})

TOOLS SENT TO MODEL:

['cancel_booking', 'check_order_status', 'check_showtimes']

### 9. ToolErrorMiddleware
* Catch exceptions raised during tool execution and convert them into error `ToolMessages` instead of halting the agent run. 
* Example Use:
    * Letting the model retry a failed tool call with corrected arguments.
    * Surfacing controlled, sanitized error messages instead of raw exception details.
    * Preventing unexpected tool exceptions from crashing the agent

* `lookup_seat_map` tool is designed to fail if the seat number format is wrong, demonstrating how the `ToolErrorMiddleware` can catch and handle tool errors gracefully.
* The middleware can log the error, provide a user-friendly message, or even suggest corrections.

In [29]:
@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."

* This method is used as callback for the ToolErrorMiddleware to handle errors from the lookup_seat_map tool.
* This helps to provide user-friendly error messages instead of breaking the agent.

In [30]:

def on_seat_error(exc: Exception, request) -> str | None:
    if isinstance(exc, ValueError):
        # Return the EXCEPTION TYPE, not str(exc) -- internal detail never reaches the model
        return f"`{request.tool_call['name']}` failed with {type(exc).__name__}. Please provide a valid seat number like 'A12'."
    return None  # anything else propagates and halts the run


In [ ]:
error_handled_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=tools,
    middleware=[ToolErrorMiddleware(on_error=on_seat_error)],
)

* Invoking agent to call `lookup_seat_map` Tool that looks for wrong Seat.
* Without `ToolErrorMiddleware` It would have failed with below error
```python 
    ValueError: Malformed seat number '12' -- expected a letter+number like 'A12'.
```
* With `ToolErrorMiddleware` this converts error to `ToolMessage`: `lookup_seat_map` failed with ValueError. Please provide a valid seat number like 'A12'.
* Then this `ToolMessage` is passed on to LLM  and proper issue is sent back to user gracefully without breaking the Agent flow.

In [32]:
result = error_handled_agent.invoke({"messages": [("user", "Look up seat 12 for Dune Part Two")]})
print_messages(result)

HumanMessage: Look up seat 12 for Dune Part Two

================================================================================

Agent: AIMessage with tool calls:

Tool Call 1 Name: lookup_seat_map
Tool Call 1 Args: {'movie_title': 'Dune Part Two', 'seat_number': '12'}

================================================================================

ToolMessage: `lookup_seat_map` failed with ValueError. Please provide a valid seat number like 'A12'.

================================================================================

Agent: The seat number must include a row letter, like A12 or B7. Please provide a seat in that format for Dune 
Part Two (e.g., A12, C7, etc.).

If you’d like, I can try a few common options for you (e.g., A12, A7, B5). Which seat would you like me to check?

================================================================================

### 10 ToolRetryMiddleware
* Automatically retry failed tool calls with configurable exponential backoff (Wait duration between call).
* Exaple Use:
    * Handling transient failures in external API calls.
    * Improving reliability of network-dependent tools.

**Configurable Options**
* `max_retries`: default = 2
* `tools`: Optional list of tools or tool names to apply retry logic to
* `retry_on`: Either a tuple of exception types to retry on, or a callable that takes an exception and returns True if it should be retried.
* `on_failure`: string | callable - `default`:"continue"  ['continue', 'error', ''Callable' Function that takes the exception and returns a string for the ToolMessage content]
* `backoff_factor`: number - default:"2.0" : Multiplier for exponential backoff. <br>Each retry waits `initial_delay * (backoff_factor ** retry_number)` seconds. Set to 0.0 for constant delay.
* `initial_delay`: number - default:"1.0" - Initial delay in seconds before first retry

**Below is example how `backoff_factor` is set for each retry**

In [39]:
initial_delay=1.0
backoff_factor=2.0

# delay = initial_delay * backoff_factor ** retry_number
print("Backoff for Retry 0:  ", initial_delay * backoff_factor ** 0) 
print("Backoff for Retry 1:  ", initial_delay * backoff_factor ** 1) 
print("Backoff for Retry 2:  ", initial_delay * backoff_factor ** 2)

Backoff for Retry 0:   1.0

Backoff for Retry 1:   2.0

Backoff for Retry 2:   4.0

* This method used to demonstrate a resilient tool agent with retry middleware.
* Method is to simulate a flaky external service that can fail transiently, to demonstrate the retry middleware.
* This is a tool that will randomly fail, and the retry middleware will handle the retries with exponential backoff.
* Also it will print the time since the last call, to show the backoff in action.

In [43]:
import time

last_called = None
@tool
def flaky_showtime_check_time(movie_title: str) -> str:
    """Check showtimes via an external service that can transiently fail."""
    global last_called
    current_time = time.time()
    if last_called is not None:
        print(f"Time since last call: {current_time - last_called:.2f} seconds")
    last_called = current_time
    if not random.random() > 1:
        print("Facing Connection Error")
        raise ConnectionError("Simulated network failure")
    return f"{movie_title}: showing at 8:00 PM."

* This will invoke the flaky_showtime_check tool, and if it fails due to a ConnectionError
* It will call tool 4 times, 1st normal invoke call and 3 reties after on failure.
* This will retry up to 3 times with exponential backoff before giving up and continuing the agent's execution.

In [44]:
resilient_tool_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[flaky_showtime_check_time],
    middleware=[
        ToolRetryMiddleware(max_retries=3, backoff_factor=2.0, initial_delay=1.0, on_failure="continue"),
    ]
)

In [45]:
result = resilient_tool_agent.invoke({"messages": [("user", "Check showtimes for Interstellar, don't retry calling any tools")]})

Facing Connection Error

Time since last call: 1.06 seconds

Facing Connection Error

Time since last call: 2.22 seconds

Facing Connection Error

Time since last call: 3.14 seconds

Facing Connection Error

In [46]:
print_messages(result)

HumanMessage: Check showtimes for Interstellar, don't retry calling any tools

================================================================================

Agent: AIMessage with tool calls:

Tool Call 1 Name: flaky_showtime_check_time
Tool Call 1 Args: {'movie_title': 'Interstellar'}

================================================================================

ToolMessage: Tool 'flaky_showtime_check_time' failed after 4 attempts with ConnectionError: Simulated network 
failure. Please try again.

================================================================================

Agent: I attempted to fetch showtimes for Interstellar, but the tool failed due to a simulated network error. I did
not retry after your instruction.

Would you like me to:
- try again now,
- wait a few minutes and try again,
- or check showtimes if you tell me a city/area or preferred cinema?

================================================================================

### 11. ModelRetryMiddleware
* Automatically retry failed model calls with configurable exponential backoff.
* Example Use:
    * Handling transient failures in model API calls.
    * Improving reliability of network-dependent model requests.


In [47]:
resilient_model_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[flaky_showtime_check_time],
    middleware=[
        ModelRetryMiddleware(max_retries=3, backoff_factor=2.0, initial_delay=1.0),
    ]
)

### 12. LLMToolEmulator
* Emulate tool execution using an LLM for testing purposes, replacing actual tool calls with AI-generated responses.
* Example Use:
    * Testing agent behavior without executing real tools.
    * Developing agents when external tools are unavailable or expensive.
    * Prototyping agent workflows before implementing actual tools.

In [48]:
emulated_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=tools,
    middleware=[LLMToolEmulator(tools=["book_seats", "cancel_booking"], model="openai:gpt-5-nano")]
)
result = emulated_agent.invoke({"messages": [("user", "Book 2 seats for Interstellar")]})


In [49]:
print_messages(result)

HumanMessage: Book 2 seats for Interstellar

================================================================================

Agent: AIMessage with tool calls:

Tool Call 1 Name: book_seats
Tool Call 1 Args: {'movie_title': 'Interstellar', 'seat_count': 2}

================================================================================

ToolMessage: {
  "status": "confirmed",
  "booking_id": "INT-TRX-862104",
  "movie_title": "Interstellar",
  "seats": ["Row G, Seat 12", "Row G, Seat 13"],
  "showtime": "2026-08-29 20:15",
  "theater": "Regal Cinemas Downtown",
  "pricing": {
    "currency": "USD",
    "per_ticket": 15.00,
    "total": 30.00
  },
  "notes": "This booking is final. No changes or cancellations once confirmed.",
  "tickets_url": "https://examplecinema.local/tickets/INT-TRX-862104"
}

================================================================================

Agent: Your booking is confirmed.

- Booking ID: INT-TRX-862104
- Movie: Interstellar
- Theater: Regal Cinemas Downtown
- Showtime: 2026-08-29 at 20:15
- Seats: Row G, Seat 12 and Row G, Seat 13
- Tickets: 2 x USD 15.00 = USD 30.00
- Ticket link: https://examplecinema.local/tickets/INT-TRX-862104
- Note: This booking is final. No changes or cancellations once confirmed.

Would you like me to:
- Add this to your calendar or set a reminder?
- Email or print the ticket?
- Look up another showtime or movie?

================================================================================